In [ ]:
"""
COMPLETE SHAP ANALYSIS FOR XGBOOST MODELS (TRACK 1 & TRACK 2) - FIXED VERSION

Purpose: Post-hoc interpretability of saved XGBoost models using SHAP
"""

import os
import sys
import numpy as np
import pandas as pd
import joblib
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# SHAP
import shap

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)

# ======================================================
# 1. CONFIGURATION - UPDATE THESE PATHS FOR YOUR SYSTEM
# ======================================================

CONFIG = {
    # Model paths (from your saved models)
    'track1_model_path': 'E:/Abroad period research/New idea for 2026/Corn Yield Estimation/final_models_for_shap/XGBoost_model.pkl',  # Track 1 model
    'track2_model_path': r'E:\Abroad period research\New idea for 2026\Corn Yield Estimation\final_models_for_shap_T2\XGBoost_E1_model.pkl',  # Track 2 model
    
    # Feature info paths (from your saved models)
    'track1_features_path': 'E:/Abroad period research/New idea for 2026/Corn Yield Estimation/final_models_for_shap/XGBoost_features.json',
    'track2_features_path': 'E:/Abroad period research/New idea for 2026/Corn Yield Estimation/final_models_for_shap_T2/XGBoost_E1_features.json',
    
    # Data paths (from your dataset)
    'train_path': "E:/Abroad period research/New idea for 2026/Corn Yield Estimation/yield_train_FE_AAO.csv",
    'test_path': "E:/Abroad period research/New idea for 2026/Corn Yield Estimation/yield_test_FE_AAO.csv",
    
    # Output directories
    'output_dir': 'shap_comparison',
    'track1_output_dir': 'shap_comparison/track1',
    'track2_output_dir': 'shap_comparison/track2',
    
    # SHAP parameters
    'n_samples_for_shap': 1000,  # Number of samples for SHAP computation
    'top_n_features': 20,  # Top N features to show in plots
    
    # Plotting style
    'style': 'seaborn-v0_8-whitegrid',
    'color_palette': 'viridis',
    'dpi': 300,
}

# ======================================================
# 2. FEATURE GROUP DEFINITION
# ======================================================

def define_feature_groups(df):
    """
    Define feature groups based on your engineering pipeline.
    Returns a dictionary mapping group names to feature lists.
    This matches your feature engineering groups.
    """
    groups = {}
    
    # G0: Raw monthly features (baseline)
    g0_features = []
    for var in ["NDVI", "GPP", "PPT", "TMEAN", "TMIN", "TMAX", "TDMEAN", "VPDMAX"]:
        monthly_cols = [col for col in df.columns 
                       if col.startswith(f"{var}_") and col.split('_')[-1].isdigit()]
        g0_features.extend(monthly_cols)
    groups['G0'] = sorted(list(set(g0_features)))
    
    # G1: Seasonal aggregates
    g1_features = []
    for col in df.columns:
        if 'season' in col.lower():
            if any(stat in col.lower() for stat in ['_season_mean', '_season_sum', '_season_std', '_season_max']):
                g1_features.append(col)
    groups['G1'] = g1_features     
            
    # G2: Phenological phase features
    g2_features = []
    for col in df.columns:
        if any(phase in col.lower() for phase in ['early', 'peak', 'late']):
            g2_features.append(col)
    groups['G2'] = g2_features
    
    # G3: Climate stress indicators
    g3_features = []
    for col in df.columns:
        if any(term in col.lower() for term in ['range', 'stress', 'heat']):
            g3_features.append(col)
    groups['G3'] = g3_features
    
    # G4: Efficiency features
    g4_features = []
    for col in df.columns:
        if any(term in col.lower() for term in ['efficiency', 'ndvi_ppt', 'gpp_ppt']):
            g4_features.append(col)
    groups['G4'] = g4_features
    
    # G5: Spatial anomaly features
    g5_features = [col for col in df.columns if col.endswith('_z')]
    groups['G5'] = g5_features
    
    # Gs: STATIC FEATURES (for Track 2 only)
    gs_features = []
    
    # Soil properties (from your sample data)
    soil_properties = ['awc', 'aws', 'b_density', 'cec', 'clay_percent', 
                      'field_capacity', 'organic_matter', 'pH', 'saturated_hc', 
                      'sand_percent', 'wilting_point']  
    
    # Geographic features
    geographic_features = ['X', 'Y']  # Longitude, Latitude
    
    # Check which static features exist in the dataframe
    for feature in soil_properties + geographic_features:
        if feature in df.columns:
            gs_features.append(feature)
    
    # Also check for any features that don't match dynamic patterns
    dynamic_patterns = ['_1', '_2', '_3', '_4', '_5', '_6', '_7', '_8', '_9', '_10', '_11', '_12',
                       'season', 'early', 'peak', 'late', 'efficiency', 'z', 'range', 'stress']
    
    for col in df.columns:
        if col not in gs_features and col not in g0_features + g1_features + g2_features + g3_features + g4_features + g5_features:
            # Check if it looks like a static feature (no numbers, not in dynamic patterns)
            is_dynamic = any(pattern in col for pattern in dynamic_patterns)
            if not is_dynamic and col not in ['yield', 'year', 'STATE', 'GEOID']:
                gs_features.append(col)
    
    groups['Gs'] = sorted(list(set(gs_features)))
    
    # Remove duplicates and ensure features exist
    for group in groups:
        groups[group] = [f for f in groups[group] if f in df.columns]
        groups[group] = sorted(set(groups[group]))
    
    # Print summary
    print(f"\n📋 Feature group summary:")
    for group, features in groups.items():
        print(f"  {group}: {len(features)} features")
        if features:
            print(f"    Sample: {features[:3]}")
    
    return groups

# ======================================================
# 3. LOAD SAVED MODEL INFORMATION
# ======================================================

def load_model_and_features(model_path, features_path, track_name):
    """
    Load saved XGBoost model and its feature information.
    
    Parameters:
    -----------
    model_path : str
        Path to saved model file
    features_path : str
        Path to saved features JSON file
    track_name : str
        Name of track for logging
    
    Returns:
    --------
    dict : Contains model, feature_names, and metadata
    """
    print(f"\n🤖 Loading XGBoost model and features for {track_name}...")
    
    results = {}
    
    try:
        # 1. Load the model
        print(f"  Loading model from: {model_path}")
        if os.path.exists(model_path):
            model = joblib.load(model_path)
            results['model'] = model
            print(f"  ✅ Model loaded successfully")
            print(f"  Model type: {model.__class__.__name__}")
            
            # Check model feature count
            if hasattr(model, 'feature_importances_'):
                print(f"  Model expects {len(model.feature_importances_)} features")
        else:
            print(f"  ❌ Model file not found: {model_path}")
            return None
    
    except Exception as e:
        print(f"  ❌ Error loading model: {e}")
        return None
    
    try:
        # 2. Load feature information
        print(f"  Loading feature info from: {features_path}")
        if os.path.exists(features_path):
            with open(features_path, 'r') as f:
                features_info = json.load(f)
            
            results['feature_info'] = features_info
            results['feature_names'] = features_info.get('feature_names', [])
            results['n_features'] = features_info.get('n_features', 0)
            results['experiment_name'] = features_info.get('experiment_name', 'unknown')
            
            print(f"  ✅ Feature info loaded successfully")
            print(f"  Features in saved model: {len(results['feature_names'])}")
            print(f"  Experiment: {results['experiment_name']}")
            print(f"  Sample features: {results['feature_names'][:5]}")
        else:
            print(f"  ⚠️ Feature info file not found: {features_path}")
            # Try to get features from model
            if hasattr(model, 'get_booster'):
                try:
                    booster = model.get_booster()
                    feature_names = booster.feature_names
                    if feature_names:
                        results['feature_names'] = feature_names
                        results['n_features'] = len(feature_names)
                        print(f"  Extracted {len(feature_names)} features from model")
                except:
                    pass
    
    except Exception as e:
        print(f"  ❌ Error loading feature info: {e}")
    
    return results

# ======================================================
# 4. DATA PREPARATION WITH FEATURE ALIGNMENT
# ======================================================

def prepare_data_with_feature_alignment(train_path, test_path, model_feature_names, track):
    """
    Load data and align features with model's expected features.
    
    Parameters:
    -----------
    train_path : str
        Path to training data CSV
    test_path : str
        Path to test data CSV
    model_feature_names : list
        Features expected by the model
    track : str
        'track1' or 'track2'
    
    Returns:
    --------
    dict : Contains aligned X_train, X_test, y_train, y_test, feature_groups
    """
    print(f"\n📂 Loading and aligning data for {track}...")
    
    # Load data
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    print(f"  Original train shape: {train_df.shape}")
    print(f"  Original test shape: {test_df.shape}")
    
    # Define feature groups from the data
    feature_groups = define_feature_groups(train_df)
    
    # Get all possible features based on track
    if track == 'track1':
        # Track 1: Dynamic features only (G0-G5)
        all_possible_features = []
        for group in ['G0', 'G1', 'G2', 'G3', 'G4', 'G5']:
            all_possible_features.extend(feature_groups.get(group, []))
    elif track == 'track2':
        # Track 2: Dynamic + Static features (G0-G5 + Gs)
        all_possible_features = []
        for group in ['G0', 'G1', 'G2', 'G3', 'G4', 'G5', 'Gs']:
            all_possible_features.extend(feature_groups.get(group, []))
    else:
        raise ValueError("track must be 'track1' or 'track2'")
    
    all_possible_features = sorted(list(set(all_possible_features)))
    print(f"  Possible features for {track}: {len(all_possible_features)}")
    
    # Align features with model's expected features
    if model_feature_names:
        print(f"  Model expects {len(model_feature_names)} features")
        
        # Find common features between model and data
        common_features = [f for f in model_feature_names if f in train_df.columns]
        missing_in_data = [f for f in model_feature_names if f not in train_df.columns]
        extra_in_data = [f for f in all_possible_features if f not in model_feature_names]
        
        print(f"  Common features: {len(common_features)}")
        print(f"  Features in model but missing in data: {len(missing_in_data)}")
        if missing_in_data:
            print(f"    Missing: {missing_in_data[:5]}")
        print(f"  Features in data but not in model: {len(extra_in_data)}")
        
        # Use model's feature order
        final_features = [f for f in model_feature_names if f in common_features]
        
        # Check if we have enough features
        if len(final_features) < len(model_feature_names):
            print(f"  ⚠️ Warning: Only {len(final_features)}/{len(model_feature_names)} features available")
            # Add missing features with zeros
            for f in missing_in_data:
                print(f"    Will create dummy feature: {f}")
                train_df[f] = 0.0
                test_df[f] = 0.0
                final_features.append(f)
    else:
        # Use all possible features
        final_features = all_possible_features
        print(f"  Using all {len(final_features)} possible features")
    
    # Ensure we have the features in the correct order
    # final_features = sorted(list(set(final_features)))
    final_features = [f for f in model_feature_names if f in train_df.columns]

    print(f"  Final feature count: {len(final_features)}")
    print(f"  Sample features: {final_features[:5]}")
    
    # Prepare feature matrices
    X_train = train_df[final_features].copy()
    X_test = test_df[final_features].copy()
    
    # Handle missing values
    X_train = X_train.fillna(X_train.mean())
    X_test = X_test.fillna(X_test.mean())
    
    # Target variable
    y_train = train_df['yield'].values
    y_test = test_df['yield'].values
    
    print(f"  X_train aligned shape: {X_train.shape}")
    print(f"  X_test aligned shape: {X_test.shape}")
    
    return {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'feature_names': final_features,
        'feature_groups': feature_groups,
        'track': track,
        'model_feature_names': model_feature_names
    }

# ======================================================
# 5. SHAP COMPUTATION WITH SAFETY CHECKS
# ======================================================

def compute_shap_values_safely(model, X_data, feature_names, n_samples=None):
    """
    Compute SHAP values with safety checks for feature alignment.
    
    Parameters:
    -----------
    model : XGBoost model
        Trained XGBoost model
    X_data : pandas DataFrame
        Data to compute SHAP values for
    feature_names : list
        Expected feature names
    n_samples : int or None
        Number of samples to use (None = all)
    
    Returns:
    --------
    explainer : SHAP TreeExplainer
    shap_values : numpy array
        SHAP values
    X_sample : pandas DataFrame
        Sampled data used for SHAP
    """
    print("🔍 Computing SHAP values with safety checks...")
    
    # Convert to numpy array for SHAP
    if isinstance(X_data, pd.DataFrame):
        X_array = X_data.values
    else:
        X_array = X_data
    
    print(f"  Input data shape: {X_array.shape}")
    print(f"  Model expects features: {len(feature_names)}")
    
    # Check feature count match
    if X_array.shape[1] != len(feature_names):
        print(f"  ⚠️ Warning: Data has {X_array.shape[1]} features, model expects {len(feature_names)}")
        print(f"  Attempting to align features...")
        
        # If we have more features than model expects, use first n features
        if X_array.shape[1] > len(feature_names):
            print(f"  Using first {len(feature_names)} features")
            X_array = X_array[:, :len(feature_names)]
        # If we have fewer features, pad with zeros
        elif X_array.shape[1] < len(feature_names):
            print(f"  Padding with zeros to match {len(feature_names)} features")
            padding = np.zeros((X_array.shape[0], len(feature_names) - X_array.shape[1]))
            X_array = np.hstack([X_array, padding])
    
    print(f"  Aligned data shape: {X_array.shape}")
    
    # Sample data if specified
    if n_samples is not None and n_samples < len(X_array):
        print(f"  Sampling {n_samples} samples (out of {len(X_array)})")
        indices = np.random.choice(len(X_array), min(n_samples, len(X_array)), replace=False)
        X_sample_array = X_array[indices]
        if isinstance(X_data, pd.DataFrame):
            X_sample_df = X_data.iloc[indices]
        else:
            X_sample_df = None
    else:
        X_sample_array = X_array
        X_sample_df = X_data if isinstance(X_data, pd.DataFrame) else None
        print(f"  Using all {len(X_sample_array)} samples")
    
    print(f"  Final data shape for SHAP: {X_sample_array.shape}")
    
    try:
        # Create TreeExplainer
        print("  Creating TreeExplainer...")
        explainer = shap.TreeExplainer(model)
        
        # Compute SHAP values
        print("  Computing SHAP values...")
        shap_values = explainer.shap_values(X_sample_array)
        
        print(f"  ✅ SHAP computation successful!")
        print(f"  SHAP values shape: {shap_values.shape}")
        print(f"  Expected value (base value): {explainer.expected_value:.4f}")
        
        return explainer, shap_values, X_sample_df if X_sample_df is not None else X_sample_array
        
    except Exception as e:
        print(f"  ❌ Error computing SHAP values: {e}")
        
        # Try alternative approach
        print("  Trying alternative SHAP computation...")
        try:
            explainer = shap.TreeExplainer(model, X_sample_array[:100])  # Small background
            shap_values = explainer.shap_values(X_sample_array)
            
            print(f"  ✅ Alternative SHAP computation successful!")
            print(f"  SHAP values shape: {shap_values.shape}")
            
            return explainer, shap_values, X_sample_df if X_sample_df is not None else X_sample_array
            
        except Exception as e2:
            print(f"  ❌ Alternative also failed: {e2}")
            raise

# ======================================================
# 6. GROUP-LEVEL SHAP AGGREGATION
# ======================================================

def aggregate_shap_by_groups(shap_values, feature_names, feature_groups, track):
    """
    Aggregate feature-level SHAP values to group-level importance.
    
    Parameters:
    -----------
    shap_values : numpy array
        SHAP values (n_samples, n_features)
    feature_names : list
        List of feature names
    feature_groups : dict
        Dictionary mapping group names to feature lists
    track : str
        'track1' or 'track2'
    
    Returns:
    --------
    group_importance_df : pandas DataFrame
        DataFrame with group-level SHAP importance
    feature_importance_df : pandas DataFrame
        DataFrame with feature-level SHAP importance
    """
    print("📊 Aggregating SHAP values by feature groups...")
    
    # Calculate mean absolute SHAP per feature
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    
    # Create feature-level importance DataFrame
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'mean_abs_shap': mean_abs_shap,
        'group': 'Other'  # Default group
    })
    
    # Assign features to groups
    for group_name, group_features in feature_groups.items():
        # Only include groups relevant to this track
        if track == 'track1' and group_name == 'Gs':
            continue
            
        for feature in group_features:
            if feature in feature_names:
                feature_importance_df.loc[feature_importance_df['feature'] == feature, 'group'] = group_name
    
    # Calculate group-level importance
    group_importance = {}
    group_counts = {}
    group_features_dict = {}
    
    for group_name in feature_groups.keys():
        # Only include groups relevant to this track
        if track == 'track1' and group_name == 'Gs':
            continue
            
        # Get features in this group
        group_features = [f for f in feature_groups[group_name] if f in feature_names]
        group_features_dict[group_name] = group_features
        
        if group_features:
            # Get indices of these features
            indices = [feature_names.index(f) for f in group_features]
            # Calculate mean absolute SHAP for this group
            group_shap_values = mean_abs_shap[indices]
            group_importance[group_name] = group_shap_values.sum()
            group_counts[group_name] = len(group_features)
        else:
            group_importance[group_name] = 0.0
            group_counts[group_name] = 0
    
    # Create group importance DataFrame
    group_importance_df = pd.DataFrame({
        'group': list(group_importance.keys()),
        'total_shap_importance': list(group_importance.values()),
        'n_features': [group_counts[g] for g in group_importance.keys()],
        'features': [group_features_dict[g] for g in group_importance.keys()]
    })
    
    # Calculate normalized importance
    total_importance = group_importance_df['total_shap_importance'].sum()
    if total_importance > 0:
        group_importance_df['normalized_importance'] = group_importance_df['total_shap_importance'] / total_importance * 100
    else:
        group_importance_df['normalized_importance'] = 0.0
    
    # Sort by importance
    group_importance_df = group_importance_df.sort_values('total_shap_importance', ascending=False)
    feature_importance_df = feature_importance_df.sort_values('mean_abs_shap', ascending=False)
    
    print(f"  Found {len(group_importance_df)} feature groups")
    print(f"  Total SHAP importance: {total_importance:.4f}")
    
    # Print group summary
    print(f"\n  Group importance summary:")
    for _, row in group_importance_df.iterrows():
        if row['total_shap_importance'] > 0:
            print(f"    {row['group']}: {row['normalized_importance']:.1f}% ({row['n_features']} features)")
    
    return group_importance_df, feature_importance_df

# ======================================================
# 7. VISUALIZATION FUNCTIONS (IMPROVED)
# ======================================================

def setup_plotting_style():
    """Set up publication-quality plotting style."""
    plt.style.use('seaborn-v0_8-whitegrid')
    mpl.rcParams['figure.figsize'] = [10, 6]
    mpl.rcParams['figure.dpi'] = 300
    mpl.rcParams['savefig.dpi'] = 300
    mpl.rcParams['font.size'] = 11
    mpl.rcParams['axes.titlesize'] = 14
    mpl.rcParams['axes.labelsize'] = 12
    mpl.rcParams['xtick.labelsize'] = 10
    mpl.rcParams['ytick.labelsize'] = 10
    mpl.rcParams['legend.fontsize'] = 10
    mpl.rcParams['figure.titlesize'] = 16

def plot_shap_summary(shap_values, feature_names, X_data, track, output_dir, top_n=20):
    """
    Create SHAP summary plot (beeswarm plot).
    """
    print(f"📈 Creating SHAP summary plot for {track}...")
    
    # Limit to top N features for clarity
    if len(feature_names) > top_n:
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        top_indices = np.argsort(mean_abs_shap)[-top_n:][::-1]
        
        shap_values_top = shap_values[:, top_indices]
        feature_names_top = [feature_names[i] for i in top_indices]
        
        if isinstance(X_data, pd.DataFrame):
            X_data_top = X_data.iloc[:, top_indices]
        else:
            X_data_top = X_data[:, top_indices]
    else:
        shap_values_top = shap_values
        feature_names_top = feature_names
        X_data_top = X_data
    
    # Create plot
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Create SHAP summary plot
    shap.summary_plot(
        shap_values_top,
        X_data_top,
        feature_names=feature_names_top,
        show=False,
        max_display=top_n,
        plot_size=None,
        color_bar_label='Feature value',
        alpha=0.7
    )
    
    # Customize plot
    plt.title(f'SHAP Summary Plot', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('SHAP value (impact on model output)', fontsize=12)
    
    # Adjust layout
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(output_dir, f'shap_summary_{track}.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  ✅ SHAP summary plot saved to: {plot_path}")
    
    # Also create a bar plot version
    plot_shap_bar(shap_values, feature_names, track, output_dir, top_n)

def plot_shap_bar(shap_values, feature_names, track, output_dir, top_n=20):
    """
    Create SHAP bar plot (feature importance).
    """
    # Calculate mean absolute SHAP
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    
    # Get top N features
    top_indices = np.argsort(mean_abs_shap)[-top_n:][::-1]
    top_features = [feature_names[i] for i in top_indices]
    top_shap = mean_abs_shap[top_indices]
    
    # Create plot
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Create horizontal bar plot
    y_pos = np.arange(len(top_features))
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_features)))
    bars = ax.barh(y_pos, top_shap, color=colors)
    
    # Customize plot
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_features, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel('Mean |SHAP value| (average impact magnitude)', fontsize=12)
    ax.set_title(f'Feature Importance (SHAP) - {track.upper()}\n(Top {top_n} Features)', 
                 fontsize=16, fontweight='bold', pad=20)
    
    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars, top_shap)):
        width = bar.get_width()
        ax.text(width * 1.01, bar.get_y() + bar.get_height()/2, f'{val:.4f}', 
                va='center', fontsize=9)
    
    # Add grid
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(output_dir, f'shap_bar_{track}.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  ✅ SHAP bar plot saved to: {plot_path}")

def plot_group_importance(group_importance_df, track, output_dir):
    """
    Create group-level importance bar plot.
    """
    print(f"📊 Creating group-level importance plot for {track}...")
    
    # Filter out groups with zero importance
    plot_df = group_importance_df[group_importance_df['total_shap_importance'] > 0].copy()
    
    if plot_df.empty:
        print("  ⚠️ No groups with importance > 0 to plot")
        return
    
    # Sort by importance
    plot_df = plot_df.sort_values('total_shap_importance', ascending=True)
    
    # Create color palette
    colors = plt.cm.Set3(np.linspace(0, 1, len(plot_df)))
    
    # Create plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    # Plot 1: Total SHAP importance
    bars1 = ax1.barh(plot_df['group'], plot_df['total_shap_importance'], color=colors)
    ax1.set_xlabel('Total SHAP Importance (Σ|SHAP|)', fontsize=12)
    ax1.set_title(f'Group-Level SHAP Importance - {track.upper()}\n(Sum of Absolute SHAP Values)', 
                  fontsize=14, fontweight='bold', pad=20)
    
    # Add value labels
    for bar in bars1:
        width = bar.get_width()
        ax1.text(width * 1.01, bar.get_y() + bar.get_height()/2, f'{width:.3f}', 
                va='center', fontsize=10)
    
    # Plot 2: Normalized importance
    bars2 = ax2.barh(plot_df['group'], plot_df['normalized_importance'], color=colors)
    ax2.set_xlabel('Normalized Importance (%)', fontsize=12)
    ax2.set_title(f'Group-Level Relative Importance - {track.upper()}\n(Percentage of Total)', 
                  fontsize=14, fontweight='bold', pad=20)
    
    # Add value labels
    for bar in bars2:
        width = bar.get_width()
        ax2.text(width * 1.01, bar.get_y() + bar.get_height()/2, f'{width:.1f}%', 
                va='center', fontsize=10)
    
    # Add grid to both plots
    ax1.grid(True, alpha=0.3, axis='x')
    ax2.grid(True, alpha=0.3, axis='x')
    
    # Add number of features as text in bars
    for i, (bar1, bar2, row) in enumerate(zip(bars1, bars2, plot_df.itertuples())):
        # Add feature count to first bar
        ax1.text(bar1.get_width() * 0.1, bar1.get_y() + bar1.get_height()/2, 
                f'n={row.n_features}', va='center', fontsize=9, color='white', fontweight='bold')
    
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(output_dir, f'group_importance_{track}.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  ✅ Group importance plot saved to: {plot_path}")

# ======================================================
# 8. MAIN ANALYSIS FUNCTION FOR EACH TRACK
# ======================================================

def analyze_track_complete(track_name, config):
    """
    Complete SHAP analysis for a single track.
    
    Parameters:
    -----------
    track_name : str
        'track1' or 'track2'
    config : dict
        Configuration dictionary
    
    Returns:
    --------
    dict : Analysis results
    """
    print(f"\n{'='*80}")
    print(f"🚀 STARTING COMPLETE SHAP ANALYSIS FOR {track_name.upper()}")
    print(f"{'='*80}")
    
    # Set paths based on track
    if track_name == 'track1':
        model_path = config['track1_model_path']
        features_path = config['track1_features_path']
        output_dir = config['track1_output_dir']
    elif track_name == 'track2':
        model_path = config['track2_model_path']
        features_path = config['track2_features_path']
        output_dir = config['track2_output_dir']
    else:
        raise ValueError("track_name must be 'track1' or 'track2'")
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Load model and feature information
    model_info = load_model_and_features(model_path, features_path, track_name)
    if model_info is None:
        print(f"❌ Failed to load model for {track_name}")
        return None
    
    # 2. Prepare data with feature alignment
    data_dict = prepare_data_with_feature_alignment(
        config['train_path'],
        config['test_path'],
        model_info.get('feature_names', []),
        track_name
    )
    
    # 3. Compute SHAP values with safety checks
    explainer, shap_values, X_sample = compute_shap_values_safely(
        model_info['model'],
        data_dict['X_test'],
        data_dict['feature_names'],
        n_samples=config['n_samples_for_shap']
    )
    
    # 4. Aggregate by groups
    group_importance_df, feature_importance_df = aggregate_shap_by_groups(
        shap_values, 
        data_dict['feature_names'], 
        data_dict['feature_groups'], 
        track_name
    )
    
    # 5. Save results
    print("\n💾 Saving analysis results...")
    
    # Save SHAP values
    shap_values_path = os.path.join(output_dir, 'shap_values.npy')
    np.save(shap_values_path, shap_values)
    print(f"  ✅ SHAP values saved to: {shap_values_path}")
    
    # Save group importance
    group_csv_path = os.path.join(output_dir, 'group_importance.csv')
    # Don't save the 'features' column as it contains lists
    group_to_save = group_importance_df.drop('features', axis=1) if 'features' in group_importance_df.columns else group_importance_df
    group_to_save.to_csv(group_csv_path, index=False, float_format='%.6f')
    print(f"  ✅ Group importance saved to: {group_csv_path}")
    
    # Save feature importance
    feature_csv_path = os.path.join(output_dir, 'feature_importance.csv')
    feature_importance_df.to_csv(feature_csv_path, index=False, float_format='%.6f')
    print(f"  ✅ Feature importance saved to: {feature_csv_path}")
    
    # Save metadata
    metadata_path = os.path.join(output_dir, 'analysis_metadata.json')
    with open(metadata_path, 'w') as f:
        metadata = {
            'track': track_name,
            'model_path': model_path,
            'n_features_analyzed': len(data_dict['feature_names']),
            'n_samples_shap': shap_values.shape[0],
            'expected_value': float(explainer.expected_value),
            'shap_values_shape': list(shap_values.shape),
            'feature_groups': {k: len(v) for k, v in data_dict['feature_groups'].items()},
            'timestamp': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        json.dump(metadata, f, indent=4)
    print(f"  ✅ Metadata saved to: {metadata_path}")
    
    # 6. Create visualizations
    print("\n🎨 Creating visualizations...")
    setup_plotting_style()
    
    # SHAP summary plot
    plot_shap_summary(
        shap_values, data_dict['feature_names'], X_sample,
        track_name, output_dir, top_n=min(config['top_n_features'], len(data_dict['feature_names']))
    )
    
    # Group importance plot
    plot_group_importance(group_importance_df, track_name, output_dir)
    
    # 7. Print summary
    print(f"\n📋 COMPLETE SUMMARY FOR {track_name.upper()}:")
    print(f"  Model: {model_info.get('experiment_name', 'Unknown experiment')}")
    print(f"  Features analyzed: {len(data_dict['feature_names'])}")
    print(f"  Samples used for SHAP: {shap_values.shape[0]}")
    print(f"  Feature groups: {len(group_importance_df)}")
    
    if len(group_importance_df) > 0:
        top_group = group_importance_df.iloc[0]
        print(f"  Most important group: {top_group['group']} ({top_group['normalized_importance']:.1f}%)")
    
    if len(feature_importance_df) > 0:
        top_feature = feature_importance_df.iloc[0]
        print(f"  Most important feature: {top_feature['feature']} (SHAP = {top_feature['mean_abs_shap']:.4f})")
    
    print(f"  Results saved in: {output_dir}")
    
    return {
        'model': model_info['model'],
        'explainer': explainer,
        'shap_values': shap_values,
        'X_sample': X_sample,
        'group_importance': group_importance_df,
        'feature_importance': feature_importance_df,
        'feature_names': data_dict['feature_names'],
        'feature_groups': data_dict['feature_groups'],
        'track': track_name,
        'metadata': metadata
    }

# ======================================================
# 9. COMPARISON AND REPORT GENERATION
# ======================================================

def create_comparison_report(track1_results, track2_results, output_dir):
    """
    Create comparison report between Track 1 and Track 2.
    """
    print(f"\n{'='*80}")
    print(f"📊 CREATING COMPARISON REPORT")
    print(f"{'='*80}")
    
    if track1_results is None or track2_results is None:
        print("❌ Cannot create comparison report: missing results")
        return
    
    # Create comparison DataFrame
    comparison_data = []
    
    # Track 1 groups
    for _, row in track1_results['group_importance'].iterrows():
        if row['total_shap_importance'] > 0:
            comparison_data.append({
                'track': 'Track 1 (Dynamic only)',
                'group': row['group'],
                'importance': row['normalized_importance'],
                'n_features': row['n_features']
            })
    
    # Track 2 groups (excluding Gs if it's 0)
    for _, row in track2_results['group_importance'].iterrows():
        if row['total_shap_importance'] > 0:
            comparison_data.append({
                'track': 'Track 2 (Dynamic + Static)',
                'group': row['group'],
                'importance': row['normalized_importance'],
                'n_features': row['n_features']
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Save comparison data
    comparison_path = os.path.join(output_dir, 'track_comparison.csv')
    comparison_df.to_csv(comparison_path, index=False, float_format='%.2f')
    print(f"✅ Comparison data saved to: {comparison_path}")
    
    # Create comparison plot
    create_comparison_plot(comparison_df, output_dir)
    
    # Generate detailed report
    generate_detailed_report(track1_results, track2_results, output_dir)

def create_comparison_plot(comparison_df, output_dir):
    """
    Create comparison plot between tracks.
    """
    # Pivot for plotting
    pivot_df = comparison_df.pivot_table(index='group', columns='track', 
                                        values='importance', fill_value=0)
    
    # Sort by Track 2 importance
    if 'Track 2 (Dynamic + Static)' in pivot_df.columns:
        pivot_df = pivot_df.sort_values('Track 2 (Dynamic + Static)', ascending=False)
    
    # Create plot
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Plot settings
    bar_width = 0.35
    x = np.arange(len(pivot_df))
    
    # Create bars
    if 'Track 1 (Dynamic only)' in pivot_df.columns:
        bars1 = ax.bar(x - bar_width/2, pivot_df['Track 1 (Dynamic only)'], 
                       bar_width, label='Track 1 (Dynamic only)', alpha=0.8, color='steelblue')
    
    if 'Track 2 (Dynamic + Static)' in pivot_df.columns:
        bars2 = ax.bar(x + bar_width/2, pivot_df['Track 2 (Dynamic + Static)'], 
                       bar_width, label='Track 2 (Dynamic + Static)', alpha=0.8, color='darkorange')
    
    # Customize plot
    ax.set_xlabel('Feature Group', fontsize=12)
    ax.set_ylabel('Normalized Importance (%)', fontsize=12)
    ax.set_title('Feature Group Importance Comparison\nTrack 1 vs Track 2', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(pivot_df.index, rotation=45, ha='right', fontsize=10)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bars in [bars1, bars2] if 'bars1' in locals() and 'bars2' in locals() else []:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                       f'{height:.1f}%', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(output_dir, 'track_comparison.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Comparison plot saved to: {plot_path}")

def generate_detailed_report(track1_results, track2_results, output_dir):
    """
    Generate a detailed text report.
    """
    report_path = os.path.join(output_dir, 'shap_analysis_report.txt')
    
    with open(report_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("SHAP ANALYSIS REPORT - TRACK 1 & TRACK 2\n")
        f.write("="*80 + "\n\n")
        
        f.write("1. TRACK 1 (Dynamic features only)\n")
        f.write("-"*40 + "\n")
        f.write(f"Features analyzed: {len(track1_results['feature_names'])}\n")
        f.write(f"Samples used for SHAP: {track1_results['shap_values'].shape[0]}\n")
        f.write(f"Expected value (base): {track1_results['explainer'].expected_value:.4f}\n\n")
        
        f.write("Group importance:\n")
        for _, row in track1_results['group_importance'].iterrows():
            if row['total_shap_importance'] > 0:
                f.write(f"  {row['group']}: {row['normalized_importance']:.1f}% ({row['n_features']} features)\n")
        
        f.write("\nTop 5 features:\n")
        for i, (_, row) in enumerate(track1_results['feature_importance'].head(5).iterrows()):
            f.write(f"  {i+1}. {row['feature']}: {row['mean_abs_shap']:.4f}\n")
        
        f.write("\n\n2. TRACK 2 (Dynamic + Static features)\n")
        f.write("-"*40 + "\n")
        f.write(f"Features analyzed: {len(track2_results['feature_names'])}\n")
        f.write(f"Samples used for SHAP: {track2_results['shap_values'].shape[0]}\n")
        f.write(f"Expected value (base): {track2_results['explainer'].expected_value:.4f}\n\n")
        
        f.write("Group importance:\n")
        for _, row in track2_results['group_importance'].iterrows():
            if row['total_shap_importance'] > 0:
                f.write(f"  {row['group']}: {row['normalized_importance']:.1f}% ({row['n_features']} features)\n")
        
        f.write("\nTop 5 features:\n")
        for i, (_, row) in enumerate(track2_results['feature_importance'].head(5).iterrows()):
            f.write(f"  {i+1}. {row['feature']}: {row['mean_abs_shap']:.4f}\n")
        
        f.write("\n\n3. KEY INSIGHTS\n")
        f.write("-"*40 + "\n")
        
        # Static feature analysis
        if 'Gs' in track2_results['group_importance']['group'].values:
            static_row = track2_results['group_importance'][track2_results['group_importance']['group'] == 'Gs'].iloc[0]
            if static_row['total_shap_importance'] > 0:
                f.write(f"• Static features (Gs) contribute {static_row['normalized_importance']:.1f}% to model predictions\n")
                f.write(f"  Number of static features: {static_row['n_features']}\n")
        
        # Comparison insights
        f.write("\n• Feature count comparison:\n")
        f.write(f"  Track 1: {len(track1_results['feature_names'])} features\n")
        f.write(f"  Track 2: {len(track2_results['feature_names'])} features\n")
        f.write(f"  Additional features in Track 2: {len(track2_results['feature_names']) - len(track1_results['feature_names'])}\n")
        
        f.write("\n" + "="*80 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*80 + "\n")
    
    print(f"✅ Detailed report saved to: {report_path}")

# ======================================================
# 10. MAIN EXECUTION FUNCTION
# ======================================================

def main_complete():
    """
    Complete main function to run SHAP analysis for both tracks.
    """
    print("="*80)
    print("🎯 COMPLETE XGBOOST SHAP ANALYSIS - TRACK 1 & TRACK 2")
    print("="*80)
    print("This script will:")
    print("  1. Load saved XGBoost models for both tracks")
    print("  2. Align features between models and data")
    print("  3. Compute SHAP values for test data")
    print("  4. Aggregate importance by feature groups")
    print("  5. Generate publication-ready visualizations")
    print("  6. Create comparison reports")
    print("="*80)
    
    # Create output directories
    os.makedirs(CONFIG['output_dir'], exist_ok=True)
    os.makedirs(CONFIG['track1_output_dir'], exist_ok=True)
    os.makedirs(CONFIG['track2_output_dir'], exist_ok=True)
    
    # Check if files exist
    print("\n🔍 Checking required files...")
    
    required_files = [
        ('Track 1 model', CONFIG['track1_model_path']),
        ('Track 1 features', CONFIG['track1_features_path']),
        ('Track 2 model', CONFIG['track2_model_path']),
        ('Track 2 features', CONFIG['track2_features_path']),
        ('Training data', CONFIG['train_path']),
        ('Test data', CONFIG['test_path'])
    ]
    
    all_files_exist = True
    for name, path in required_files:
        if os.path.exists(path):
            print(f"  ✅ {name}: Found")
        else:
            print(f"  ❌ {name}: Not found at {path}")
            all_files_exist = False
    
    if not all_files_exist:
        print("\n⚠️ Some required files are missing. Please check the paths.")
        return
    
    # Run analysis for Track 1
    print("\n" + "="*80)
    print("ANALYZING TRACK 1...")
    print("="*80)
    
    track1_results = analyze_track_complete('track1', CONFIG)
    
    if track1_results is None:
        print("❌ Failed to analyze Track 1")
        return
    
    # Run analysis for Track 2
    print("\n" + "="*80)
    print("ANALYZING TRACK 2...")
    print("="*80)
    
    track2_results = analyze_track_complete('track2', CONFIG)
    
    if track2_results is None:
        print("❌ Failed to analyze Track 2")
        # Continue with just Track 1 results
        print("\n⚠️ Continuing with Track 1 results only...")
        track2_results = None
    
    # Create comparison report if both tracks were successful
    if track1_results is not None and track2_results is not None:
        create_comparison_report(track1_results, track2_results, CONFIG['output_dir'])
    
    # Final summary
    print("\n" + "="*80)
    print("🎉 ANALYSIS COMPLETE!")
    print("="*80)
    
    print(f"\n📁 RESULTS SAVED IN:")
    print(f"  Track 1: {CONFIG['track1_output_dir']}")
    print(f"  Track 2: {CONFIG['track2_output_dir']}")
    print(f"  Comparisons: {CONFIG['output_dir']}")
    
    print(f"\n📋 FILES GENERATED FOR EACH TRACK:")
    print("  1. shap_values.npy - Raw SHAP values")
    print("  2. group_importance.csv - Group-level importance")
    print("  3. feature_importance.csv - Feature-level importance")
    print("  4. shap_summary_[track].png - SHAP beeswarm plot")
    print("  5. shap_bar_[track].png - Feature importance bar plot")
    print("  6. group_importance_[track].png - Group importance plot")
    print("  7. analysis_metadata.json - Analysis metadata")
    
    if track1_results is not None and track2_results is not None:
        print(f"\n📊 COMPARISON FILES:")
        print("  1. track_comparison.csv - Numerical comparison")
        print("  2. track_comparison.png - Visual comparison")
        print("  3. shap_analysis_report.txt - Detailed report")
    
    print("\n" + "="*80)
    print("✅ READY FOR PAPER SUBMISSION!")
    print("="*80)

# ======================================================
# 11. QUICK ANALYSIS FUNCTIONS
# ======================================================

def quick_track_analysis(track_name):
    """
    Quick analysis for a single track.
    """
    if track_name not in ['track1', 'track2']:
        print("❌ Invalid track name. Use 'track1' or 'track2'.")
        return
    
    print(f"\n⚡ QUICK ANALYSIS FOR {track_name.upper()}")
    print("="*60)
    
    results = analyze_track_complete(track_name, CONFIG)
    
    if results is not None:
        print(f"\n📋 QUICK SUMMARY FOR {track_name.upper()}:")
        print(f"  Features: {len(results['feature_names'])}")
        print(f"  Samples used: {results['shap_values'].shape[0]}")
        
        print(f"\n  Top 5 features by SHAP importance:")
        for i, (_, row) in enumerate(results['feature_importance'].head(5).iterrows()):
            print(f"    {i+1}. {row['feature']}: {row['mean_abs_shap']:.4f}")
        
        print(f"\n  Top 3 groups by SHAP importance:")
        for i, (_, row) in enumerate(results['group_importance'].head(3).iterrows()):
            if row['total_shap_importance'] > 0:
                print(f"    {row['group']}: {row['normalized_importance']:.1f}%")
    
    return results

# ======================================================
# 12. DIRECT EXECUTION
# ======================================================

if __name__ == "__main__":
    """
    Entry point for running the complete SHAP analysis.
    """
    print("="*80)
    print("XGBOOST SHAP ANALYSIS - COMPLETE SOLUTION")
    print("="*80)
    print("\nOptions:")
    print("  1. Run complete analysis for both tracks (main_complete())")
    print("  2. Quick analysis for Track 1 (quick_track_analysis('track1'))")
    print("  3. Quick analysis for Track 2 (quick_track_analysis('track2'))")
    print("\nRunning complete analysis...")
    print("="*80)
    
    try:
        main_complete()
    except Exception as e:
        print(f"\n❌ Error in main analysis: {e}")
        import traceback
        traceback.print_exc()
        
        print(f"\n⚠️ Trying individual track analysis...")
        print(f"\n{'='*60}")
        print("ANALYZING TRACK 1...")
        print("="*60)
        try:
            track1_results = quick_track_analysis('track1')
        except Exception as e1:
            print(f"❌ Failed Track 1: {e1}")
        
        print(f"\n{'='*60}")
        print("ANALYZING TRACK 2...")
        print("="*60)
        try:
            track2_results = quick_track_analysis('track2')
        except Exception as e2:
            print(f"❌ Failed Track 2: {e2}")
        
        print(f"\n{'='*80}")
        print("ANALYSIS ATTEMPTED - CHECK OUTPUT DIRECTORIES")
        print("="*80)

XGBOOST SHAP ANALYSIS - COMPLETE SOLUTION

Options:
  1. Run complete analysis for both tracks (main_complete())
  2. Quick analysis for Track 1 (quick_track_analysis('track1'))
  3. Quick analysis for Track 2 (quick_track_analysis('track2'))

Running complete analysis...
🎯 COMPLETE XGBOOST SHAP ANALYSIS - TRACK 1 & TRACK 2
This script will:
  1. Load saved XGBoost models for both tracks
  2. Align features between models and data
  3. Compute SHAP values for test data
  4. Aggregate importance by feature groups
  5. Generate publication-ready visualizations
  6. Create comparison reports

🔍 Checking required files...
  ✅ Track 1 model: Found
  ✅ Track 1 features: Found
  ✅ Track 2 model: Found
  ✅ Track 2 features: Found
  ✅ Training data: Found
  ✅ Test data: Found

ANALYZING TRACK 1...

🚀 STARTING COMPLETE SHAP ANALYSIS FOR TRACK1

🤖 Loading XGBoost model and features for track1...
  Loading model from: E:/Abroad period research/New idea for 2026/Corn Yield Estimation/final_models_f